# 💽 Lecture 6 – Data 100, Fall 2026

[Acknowledgments Page](https://ds100.org/acknowledgements/)

## Starting Up SQL

Before we look at SQL syntax in detail, let's first get ourselves set up to run SQL queries in Jupyter.

### Approach #1: SQL Magic

**1. Load the `sql` Module.** 

Load `%%sql` cell magic to allow the Jupyter notebook to understand SQL.

> FYI: `%sql` is called line magic because it only applies to one line. We will see shortly that `%%sql` applies to an entire cell. So, it's called cell magic.

In [1]:
%load_ext sql

**2. Connect to a database.**  

Here, we connect to the SQLite database `basic_examples.db` and duckdb database `example_duck.db`.

In [2]:
%sql sqlite:///data/basic_examples.db --alias sqlite_ex

Connecting to 'sqlite_ex'

In [3]:
%sql duckdb:///data/example_duck.db --alias duckdb_ex

Connecting and switching to connection 'duckdb_ex'

If you were connecting to an "enterprise data platform" in the real world, you might do something like one of these. You don't have to understand exactly how these work: it's just an example.

```python
from sqlalchemy import create_engine

snow_engine = create_engine(
    f"snowflake://{user}:{password}@{account_identifier}")
%sql snow_engine --alias snow

db_engine = create_engine(
  url = f"databricks://token:{access_token}@{server_hostname}?" +
        f"http_path={http_path}&catalog={catalog}&schema={schema}"
)
%sql db_engine --alias db
```

<br/>

**3. Run a simple SQL query.** 

`%sql` tells Jupyter that only that line should be treated as a SQL command.

In [4]:
%sql SELECT * FROM Dragon;

Running query in 'duckdb_ex'

name,yr,cute,setting_id
toothless,2010,10,0
drogon,2011,-100,1
dragon 2,2019,0,2
puff,1963,100,3
smaug,1937,None,4
rhaegal,2011,-80,1


The `%%sql` command with two `%` signs tells Jupyter that the entire cell should be treated as a SQL command.

In [5]:
%%sql
SELECT * FROM Dragon;

Running query in 'duckdb_ex'

name,yr,cute,setting_id
toothless,2010,10,0
drogon,2011,-100,1
dragon 2,2019,0,2
puff,1963,100,3
smaug,1937,None,4
rhaegal,2011,-80,1


Here's the same simple query, this time on two different lines.

In [6]:
%%sql
SELECT *
FROM Dragon;

Running query in 'duckdb_ex'

name,yr,cute,setting_id
toothless,2010,10,0
drogon,2011,-100,1
dragon 2,2019,0,2
puff,1963,100,3
smaug,1937,None,4
rhaegal,2011,-80,1


#### Storing one-line `%sql` queries

For simple one-line queries, you can use the `%sql` command on the right-hand side of an assignment expression (as if it were any other Python statement) and save the output to a variable:

In [7]:
dragon_table = %sql SELECT * FROM Dragon
dragon_table

Running query in 'duckdb_ex'

name,yr,cute,setting_id
toothless,2010,10,0
drogon,2011,-100,1
dragon 2,2019,0,2
puff,1963,100,3
smaug,1937,None,4
rhaegal,2011,-80,1


As noted above, the result of the query is a Python variable of type `ResultSet`, more specifically:

In [8]:
type(dragon_table)

sql.run.resultset.ResultSet

You need to manually convert it to a `DataFrame` if you want to work with its contents in `Polars`:

In [9]:
dragon_df = dragon_table.PolarsDataFrame()
dragon_df

name,yr,cute,setting_id
str,i32,i32,i32
"""toothless""",2010,10,0
"""drogon""",2011,-100,1
"""dragon 2""",2019,0,2
"""puff""",1963,100,3
"""smaug""",1937,null,4
"""rhaegal""",2011,-80,1


You can configure `jupysql` to _automatically_ convert all outputs to Polars DataFrames. This can be handy if you intend all your Python-side work to be done with `Polars`, as it saves you from manually having to call `.PolarsDataFrame()` first on all outputs. 

- On the other hand, you don't get access to the original SQL `ResultSet` objects, which have a number of interesting properties and capabilities. You can learn more about those in the [jupysql documentation](https://jupysql.ploomber.io).

For now, let's turn this on so you can see how this simplified, "Polars all the way" worfklow looks like:

In [10]:
%config SqlMagic.autopolars = True

In [11]:
dragon_df = %sql SELECT * FROM Dragon
dragon_df

Running query in 'duckdb_ex'

name,yr,cute,setting_id
str,i32,i32,i32
"""toothless""",2010,10,0
"""drogon""",2011,-100,1
"""dragon 2""",2019,0,2
"""puff""",1963,100,3
"""smaug""",1937,null,4
"""rhaegal""",2011,-80,1


In [12]:
type(dragon_df)

polars.dataframe.frame.DataFrame

#### Storing output of multiple SQL lines

You can use the `variable <<` syntax in jupysql to store its output.

- Note: This will follow your `autopolars` state and store either a `sql.run.ResultState` or a Polars `DataFrame`.

In [13]:
%%sql res <<
SELECT *
FROM Dragon;

Running query in 'duckdb_ex'

In [14]:
res

name,yr,cute,setting_id
str,i32,i32,i32
"""toothless""",2010,10,0
"""drogon""",2011,-100,1
"""dragon 2""",2019,0,2
"""puff""",1963,100,3
"""smaug""",1937,null,4
"""rhaegal""",2011,-80,1


### Approach #2: `pl.read_database`

It turns out that `Polars` has a special-purpose function to parse SQL queries. We can pass in a SQL query as a string to get back a `DataFrame`. To achieve the same result as we did using cell magic above, we can do the following.

**1. Connect to a database**

In [15]:
import sqlalchemy 
import polars as pl

engine = sqlalchemy.create_engine("duckdb:///data/example_duck.db")
conn = engine.connect()

**2. Run a simple SQL query**

In [16]:
query = """
SELECT * 
FROM Dragon;
"""

df = pl.read_database(query, conn)
df

name,yr,cute,setting_id
str,i32,i32,i32
"""toothless""",2010,10,0
"""drogon""",2011,-100,1
"""dragon 2""",2019,0,2
"""puff""",1963,100,3
"""smaug""",1937,null,4
"""rhaegal""",2011,-80,1


### Approach #3 -- DuckDB can look up Python variables!

With DuckDB we can directly reference dataframe objects in our Python environment:

In [17]:
import seaborn as sns
import duckdb
mpg = pl.from_pandas(sns.load_dataset("mpg"))

In [18]:
# duckdb is smart and knows to look up the table mpg in python:
result = duckdb.query("SELECT * FROM mpg")
result

┌────────┬───────────┬──────────────┬────────────┬────────┬──────────────┬────────────┬─────────┬────────────────────────────┐
│  mpg   │ cylinders │ displacement │ horsepower │ weight │ acceleration │ model_year │ origin  │            name            │
│ double │   int64   │    double    │   double   │ int64  │    double    │   int64    │ varchar │          varchar           │
├────────┼───────────┼──────────────┼────────────┼────────┼──────────────┼────────────┼─────────┼────────────────────────────┤
│   18.0 │         8 │        307.0 │      130.0 │   3504 │         12.0 │         70 │ usa     │ chevrolet chevelle malibu  │
│   15.0 │         8 │        350.0 │      165.0 │   3693 │         11.5 │         70 │ usa     │ buick skylark 320          │
│   18.0 │         8 │        318.0 │      150.0 │   3436 │         11.0 │         70 │ usa     │ plymouth satellite         │
│   16.0 │         8 │        304.0 │      150.0 │   3433 │         12.0 │         70 │ usa     │ amc rebel sst

In [19]:
type(result)

duckdb.duckdb.DuckDBPyRelation

In [20]:
result.pl().head(3)

mpg,cylinders,displacement,horsepower,weight,acceleration,model_year,origin,name
f64,i64,f64,f64,i64,f64,i64,str,str
18.0,8,307.0,130.0,3504,12.0,70,"""usa""","""chevrolet chevelle malibu"""
15.0,8,350.0,165.0,3693,11.5,70,"""usa""","""buick skylark 320"""
18.0,8,318.0,150.0,3436,11.0,70,"""usa""","""plymouth satellite"""



---

## Tables and Schema

A **database** contains a collection of SQL **tables**. Let's connect to our "toy" database `example_duck.db` and explore the tables it stores.

In [21]:
%%sql
SELECT * FROM information_schema.tables

Running query in 'duckdb_ex'

table_catalog,table_schema,table_name,table_type,self_referencing_column_name,reference_generation,user_defined_type_catalog,user_defined_type_schema,user_defined_type_name,is_insertable_into,is_typed,commit_action,TABLE_COMMENT
str,str,str,str,str,str,str,str,str,str,str,str,str
"""example_duck""","""main""","""dish""","""BASE TABLE""",null,null,null,null,null,"""YES""","""NO""",null,null
"""example_duck""","""main""","""dragon""","""BASE TABLE""",null,null,null,null,null,"""YES""","""NO""",null,null
"""example_duck""","""main""","""scene""","""BASE TABLE""",null,null,null,null,null,"""YES""","""NO""",null,null
"""example_duck""","""main""","""setting""","""BASE TABLE""",null,null,null,null,null,"""YES""","""NO""",null,null


In [22]:
%%sql
SELECT * FROM information_schema.columns

Running query in 'duckdb_ex'

table_catalog,table_schema,table_name,column_name,ordinal_position,column_default,is_nullable,data_type,character_maximum_length,character_octet_length,numeric_precision,numeric_precision_radix,numeric_scale,datetime_precision,interval_type,interval_precision,character_set_catalog,character_set_schema,character_set_name,collation_catalog,collation_schema,collation_name,domain_catalog,domain_schema,domain_name,udt_catalog,udt_schema,udt_name,scope_catalog,scope_schema,scope_name,maximum_cardinality,dtd_identifier,is_self_referencing,is_identity,identity_generation,identity_start,identity_increment,identity_maximum,identity_minimum,identity_cycle,is_generated,generation_expression,is_updatable,COLUMN_COMMENT
str,str,str,str,i32,str,str,str,i32,i32,i32,i32,i32,i32,str,i32,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,i64,str,bool,bool,str,str,str,str,str,bool,str,str,bool,str
"""example_duck""","""main""","""dish""","""name""",1,null,"""NO""","""VARCHAR""",null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null
"""example_duck""","""main""","""dish""","""type""",2,null,"""YES""","""VARCHAR""",null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null
"""example_duck""","""main""","""dish""","""cost""",3,null,"""YES""","""INTEGER""",null,null,32,2,0,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null
"""example_duck""","""main""","""dragon""","""name""",1,null,"""NO""","""VARCHAR""",null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null
"""example_duck""","""main""","""dragon""","""yr""",2,null,"""YES""","""INTEGER""",null,null,32,2,0,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null
…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…
"""example_duck""","""main""","""setting""","""id""",1,null,"""NO""","""INTEGER""",null,null,32,2,0,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null
"""example_duck""","""main""","""setting""","""media""",2,null,"""NO""","""VARCHAR""",null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null
"""example_duck""","""main""","""setting""","""place""",3,null,"""YES""","""VARCHAR""",null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null


### Getting Schema information with SQLAlchemy 
How you list the tables varies across database platforms! For example, the statement:

```sql
SELECT * FROM information_schema.columns
```

only works on certain kinds of databases (specifically, databases that are compatible with PostgreSQL).

For example, if we wanted to get the schema for tables in SQLite, we'd have to do it differently:

In [23]:
sqlite_conn = sqlalchemy.create_engine("sqlite:///data/basic_examples.db").connect()
pl.read_database("SELECT * FROM sqlite_schema", sqlite_conn)

type,name,tbl_name,rootpage,sql
str,str,str,i64,str
"""table""","""sqlite_sequence""","""sqlite_sequence""",7,"""CREATE TABLE sqlite_sequence(n…"
"""table""","""Dragon""","""Dragon""",2,"""CREATE TABLE Dragon ( name…"
"""index""","""sqlite_autoindex_Dragon_1""","""Dragon""",3,null
"""table""","""Dish""","""Dish""",4,"""CREATE TABLE Dish ( name T…"
"""index""","""sqlite_autoindex_Dish_1""","""Dish""",5,null
"""table""","""Scene""","""Scene""",6,"""CREATE TABLE Scene ( id IN…"


In [24]:
display(pl.read_database("SELECT * FROM sqlite_schema", sqlite_conn)[[0, 1, 3, 4]])

type,name,tbl_name,rootpage,sql
str,str,str,i64,str
"""table""","""sqlite_sequence""","""sqlite_sequence""",7,"""CREATE TABLE sqlite_sequence(n…"
"""table""","""Dragon""","""Dragon""",2,"""CREATE TABLE Dragon ( name…"
"""table""","""Dish""","""Dish""",4,"""CREATE TABLE Dish ( name T…"
"""index""","""sqlite_autoindex_Dish_1""","""Dish""",5,null


Fortunately, SQLAlchemy is a library with some generic tools that are helpful regardless of what database platform you use.

In [25]:
from sqlalchemy import inspect
inspector = inspect(engine)
inspector.get_table_names()

['dish', 'dragon', 'scene', 'setting']

In [26]:
inspector.get_columns('scene')

[{'name': 'id',
  'type': Integer(),
  'nullable': False,
  'default': None,
  'autoincrement': False,
  'comment': None},
 {'name': 'biome',
  'type': String(),
  'nullable': False,
  'default': None,
  'autoincrement': False,
  'comment': None},
 {'name': 'city',
  'type': String(),
  'nullable': False,
  'default': None,
  'autoincrement': False,
  'comment': None},
 {'name': 'visitors',
  'type': Integer(),
  'nullable': True,
  'default': None,
  'autoincrement': False,
  'comment': None},
 {'name': 'created_at',
  'type': TIMESTAMP(),
  'nullable': True,
  'default': 'current_date()',
  'autoincrement': False,
  'comment': None}]

Same with SQLite

In [27]:
sqlite_engine = sqlalchemy.create_engine("sqlite:///data/basic_examples.db")
inspect(sqlite_engine).get_columns("scene")

[{'name': 'id',
  'type': INTEGER(),
  'nullable': True,
  'default': None,
  'primary_key': 1},
 {'name': 'biome',
  'type': TEXT(),
  'nullable': False,
  'default': None,
  'primary_key': 0},
 {'name': 'city',
  'type': TEXT(),
  'nullable': False,
  'default': None,
  'primary_key': 0},
 {'name': 'visitors',
  'type': INTEGER(),
  'nullable': True,
  'default': None,
  'primary_key': 0},
 {'name': 'created_at',
  'type': DATETIME(),
  'nullable': True,
  'default': "DATETIME('now')",
  'primary_key': 0}]

More advanced example of creating tables with primary and foreign key constraints:

In [28]:
%sql duckdb:///data/duckdb_example.db --alias student_db

Connecting and switching to connection 'student_db'

In [29]:
%%sql student_db

DROP TABLE IF EXISTS grade;
DROP TABLE IF EXISTS assignment;
DROP TABLE IF EXISTS student;


CREATE TABLE student (
    student_id INTEGER PRIMARY KEY,
    name VARCHAR,
    email VARCHAR
);

CREATE TABLE assignment (
    assignment_id INTEGER PRIMARY KEY,
    description VARCHAR
);

CREATE TABLE grade (
    student_id INTEGER,
    assignment_id INTEGER,
    score REAL CHECK (score > 0 AND score <= 100),
    FOREIGN KEY (student_id) REFERENCES student(student_id),
    FOREIGN KEY (assignment_id) REFERENCES assignment(assignment_id)
);

INSERT INTO student VALUES
(123, 'JoeyG', 'jegonzal@berkeley.edu'),
(456, 'NargesN', 'norouzi@berkeley.edu');

INSERT INTO assignment VALUES
(1, 'easy assignment'),
(2, 'hard assignment');

Success
bool


In [30]:
%%sql
INSERT INTO grade VALUES
(123, 1, 80),
(123, 2, 42),
(456, 2, 100);

Running query in 'student_db'

Success
bool


In [31]:
%sql SELECT * FROM grade;

Running query in 'student_db'

student_id,assignment_id,score
i32,i32,f32
123,1,80.0
123,2,42.0
456,2,100.0


<br/>

---

## Basic Queries

### `SELECT` and `FROM`
Every SQL query *must* contain a `SELECT` and `FROM` clause.

* `SELECT`: specify the column(s) to return in the output.
* `FROM`: specify the database table from which to extract data.

First, let's reconnect to our `duckdb_ex` database from earlier:

In [32]:
%sql duckdb_ex

Switching to connection 'duckdb_ex'

In [33]:
%%sql
SELECT * 
FROM Dragon;

Running query in 'duckdb_ex'

name,yr,cute,setting_id
str,i32,i32,i32
"""toothless""",2010,10,0
"""drogon""",2011,-100,1
"""dragon 2""",2019,0,2
"""puff""",1963,100,3
"""smaug""",1937,null,4
"""rhaegal""",2011,-80,1


In [34]:
%%sql
SELECT cute, yr 
FROM Dragon;

Running query in 'duckdb_ex'

cute,yr
i32,i32
10,2010
-100,2011
0,2019
100,1963
null,1937
-80,2011


### Aliasing with `AS`

In [35]:
%%sql
SELECT cute AS cuteness,
       yr AS "birth year"
FROM Dragon;

Running query in 'duckdb_ex'

cuteness,birth year
i32,i32
10,2010
-100,2011
0,2019
100,1963
null,1937
-80,2011


`AS` is technically optional, but often good practice to include!

In [36]:
%%sql
SELECT cute cuteness,
       yr "birth year"
FROM Dragon;

Running query in 'duckdb_ex'

cuteness,birth year
i32,i32
10,2010
-100,2011
0,2019
100,1963
null,1937
-80,2011


### Uniqueness with `DISTINCT`

In [37]:
%%sql
SELECT DISTINCT yr
FROM Dragon;

Running query in 'duckdb_ex'

yr
i32
1963
1937
2010
2019
2011


### Filtering with `WHERE`

In [38]:
%%sql
SELECT name, yr
FROM Dragon
WHERE cute > 0;

Running query in 'duckdb_ex'

name,yr
str,i32
"""toothless""",2010
"""puff""",1963


In [39]:
%%sql
SELECT name, cute, yr
FROM Dragon
WHERE cute > 0 OR yr > 2013;

Running query in 'duckdb_ex'

name,cute,yr
str,i32,i32
"""toothless""",10,2010
"""dragon 2""",0,2019
"""puff""",100,1963


In [40]:
%%sql
SELECT name, yr
FROM Dragon 
WHERE name IN ('puff', 'drogon');

Running query in 'duckdb_ex'

name,yr
str,i32
"""drogon""",2011
"""puff""",1963


In [41]:
%%sql
SELECT name, cute
FROM Dragon
WHERE cute IS NOT NULL;

Running query in 'duckdb_ex'

name,cute
str,i32
"""toothless""",10
"""drogon""",-100
"""dragon 2""",0
"""puff""",100
"""rhaegal""",-80


### Ordering data using `ORDER BY`

In [42]:
%%sql
SELECT *
FROM Dragon
ORDER BY cute DESC;

Running query in 'duckdb_ex'

name,yr,cute,setting_id
str,i32,i32,i32
"""puff""",1963,100,3
"""toothless""",2010,10,0
"""dragon 2""",2019,0,2
"""rhaegal""",2011,-80,1
"""drogon""",2011,-100,1
"""smaug""",1937,null,4


---

<br />
<br />

**Instructor note: return to slides!**

<br />
<br />

---

### Restricting output with `LIMIT` and `OFFSET`

In [43]:
%%sql
SELECT *
FROM Dragon
LIMIT 2;

Running query in 'duckdb_ex'

name,yr,cute,setting_id
str,i32,i32,i32
"""toothless""",2010,10,0
"""drogon""",2011,-100,1


In [44]:
%%sql
SELECT *
FROM Dragon
LIMIT 2
OFFSET 1;

Running query in 'duckdb_ex'

name,yr,cute,setting_id
str,i32,i32,i32
"""drogon""",2011,-100,1
"""dragon 2""",2019,0,2


### Sampling with `RANDOM()`
What if we wanted a random sample:

In [45]:
%%sql
SELECT *
FROM Dragon
ORDER BY RANDOM() 
LIMIT 2

Running query in 'duckdb_ex'

name,yr,cute,setting_id
str,i32,i32,i32
"""dragon 2""",2019,0,2
"""toothless""",2010,10,0


In [46]:
%%sql
SELECT * 
FROM Dragon USING SAMPLE reservoir(2 ROWS) REPEATABLE (100);

Running query in 'duckdb_ex'

name,yr,cute,setting_id
str,i32,i32,i32
"""drogon""",2011,-100,1
"""smaug""",1937,null,4
